# 10.2 Gittersuche

Wenn wir einen 3D-Drucker auf ein neues Filament einstellen, probieren wir
verschiedene Kombinationen aus Drucktemperatur und Druckgeschwindigkeit aus und
behalten die beste. Mit den Hyperparametern eines ML-Modells gehen wir genauso
vor. Bisher haben wir die Kreuzvalidierung genutzt, um ein einzelnes Modell
robuster zu bewerten. In diesem Kapitel vergleichen wir damit verschiedene
Hyperparameter und lernen dabei die Validierungsdaten und die Gittersuche
kennen.

## Hyperparameter mit Kreuzvalidierung vergleichen

Im letzten Kapitel haben wir die Kreuzvalidierung eingeführt. Ihr Ziel ist es,
eine robustere Bewertung der Modellleistung zu ermöglichen. Besonders bei der
Beurteilung und der Verbesserung der Verallgemeinerungsfähigkeit eines Modells
(Reduktion von Overfitting) ist die Kreuzvalidierung ein wertvolles Werkzeug.
In diesem Abschnitt nutzen wir die Kreuzvalidierung, um mehrere Varianten eines
Modells miteinander zu vergleichen.

Aus didaktischen Gründen verwenden wir weiterhin künstliche Daten, die mit der
Funktion `make_moons()` aus dem Modul `sklearn.datasets` erzeugt werden. Diese
speichern wir in einem Pandas-DataFrame und visualisieren sie anschließend mit
Plotly Express.

In [ ]:
import pandas as pd
import plotly.express as px
from sklearn.datasets import make_moons

X_array, y_array = make_moons(noise=0.5, n_samples=100, random_state=3)
daten = pd.DataFrame({
    'Merkmal 1': X_array[:,0],
    'Merkmal 2': X_array[:,1],
    'Wirkung': y_array
})
daten['Wirkung'] = daten['Wirkung'].astype('bool')

fig = px.scatter(daten, x='Merkmal 1', y='Merkmal 2', color='Wirkung',
    title='Künstliche Daten')
fig.show()

Als Nächstes trainieren wir einen Entscheidungsbaum. Da Entscheidungsbäume
häufig zur Überanpassung (Overfitting) neigen, entscheiden wir uns, die
Baumtiefe zu begrenzen. Aber welche Baumtiefe ist optimal? Die Baumtiefe ist ein
Hyperparameter, der vor dem Training des Modells festgelegt wird. Mithilfe der
Kreuzvalidierung können wir untersuchen, wie sich die Baumtiefe auf die
Modellqualität auswirkt. Wir testen die Baumtiefen 1 bis 6 und geben die
Scores auf den Testdaten aus, wobei wir uns mit einer for-Schleife die Arbeit
erleichtern.

Für die Aufteilung in Folds verwenden wir diesmal `StratifiedKFold`. Es
funktioniert wie `KFold`, sorgt aber zusätzlich dafür, dass in jedem Fold beide
Klassen möglichst im selben Verhältnis vorkommen wie im gesamten Datensatz. Bei
Klassifikationsaufgaben ist das meist die bessere Wahl, solange die Datenpunkte
unabhängig voneinander sind. Bei Zeitreihen dagegen muss zum Beispiel die
zeitliche Reihenfolge erhalten bleiben. Damit unsere Ergebnisse mit
dem Vorlesungsskript vergleichbar sind, setzen wir bei der Aufteilung und beim
Entscheidungsbaum `random_state=0`.

In [ ]:
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier

# Daten ins richtige Format bringen
X = daten[['Merkmal 1', 'Merkmal 2']]
y = daten['Wirkung']

# Vorbereitung der Kreuzvalidierung mit 5 Folds
kfold = # TODO: ???  (5 Folds mit gleichem Klassenverhältnis wie im gesamten Datensatz, Daten vorher mischen, reproduzierbar mit dem Wert 0)

# wiederholte Kreuzvalidierung für die Baumtiefen 1 bis 6
for max_tiefe in [1, 2, 3, 4, 5, 6]:
    modell = # TODO: ???  (Entscheidungsbaum mit der aktuellen Baumtiefe, reproduzierbar mit dem Wert 0)
    cv_results = cross_validate(modell, X, y, cv=kfold)
    test_scores = cv_results['test_score']
    print(f'Baumtiefe {max_tiefe}, Testscores: {test_scores}')

Die Ausgabe der einzelnen Testscores ist jedoch unübersichtlich. Stattdessen
berechnen wir besser den Mittelwert (Mean) und die Standardabweichung (Standard
Deviation) der Scores. Dazu importieren wir NumPy, verwenden die Funktionen
`np.mean()` und `np.std()` und passen die `print()`-Anweisung entsprechend an.

In [ ]:
import numpy as np

for max_tiefe in [1, 2, 3, 4, 5, 6]:
    modell = DecisionTreeClassifier(max_depth=max_tiefe, random_state=0)
    cv_results = cross_validate(modell, X, y, cv=kfold)
    test_scores = cv_results['test_score']
    mittelwert = np.mean(test_scores)
    streuung = np.std(test_scores)
    print(f'Baumtiefe {max_tiefe}, Testscores: Mittelwert {mittelwert:.2f}, Standardabweichung {streuung:.2f}')

Das beste Ergebnis erzielen wir mit einem Entscheidungsbaum der Tiefe 2. Diesen
könnten wir nun als finales Modell wählen. Allerdings zeigt die
Standardabweichung, wie stark die Scores von Fold zu Fold schwanken. Der
Vorsprung von 0.03 auf die Tiefe 3 ist kleiner als diese Schwankung. Solche
knappen Unterschiede sollten wir nicht überbewerten.

## Trainings-, Validierungs- und Testdaten

Im vorigen Abschnitt hat die Baumtiefe 2 mit einem mittleren Score von 0.79 am
besten abgeschnitten. Es gibt jedoch ein Problem. Der Entscheidungsbaum selbst
hat die zurückgehaltenen Daten nie zum Trainieren gesehen. Wir haben sie aber
indirekt genutzt, denn wir haben unsere Wahl der Baumtiefe genau auf diese
Scores gestützt. Damit ist der Score von 0.79 kein unbefangenes Urteil über die
Prognosequalität mehr. Wir haben aus sechs Werten den besten herausgesucht, und
der beste von sechs Werten fällt meistens etwas zu optimistisch aus.

Um die Prognosequalität ehrlich einzuschätzen, brauchen wir also Daten, die
weder das Modell noch wir selbst bei der Auswahl benutzt haben. Die Lösung
dafür ist `train_test_split()`.

Zuerst teilen wir die Daten in Trainings- und Testdaten. In der Praxis machen
wir das ganz am Anfang, noch vor dem ersten Vergleich. Im vorigen Abschnitt sind
wir nur anders vorgegangen, um das Problem sichtbar zu machen. Die Testdaten
legen wir zur Seite und rühren sie bis zum Schluss nicht mehr an. Dann verwenden
wir die Kreuzvalidierung nur auf den Trainingsdaten, um die Hyperparameter zu
bewerten. Die Kreuzvalidierung teilt die Trainingsdaten dabei erneut auf,
nämlich in einen Trainingsteil und einen zurückgehaltenen Teil. Damit wir diesen
zurückgehaltenen Teil nicht mit den richtigen Testdaten verwechseln, nennen wir
ihn **Validierungsdaten**. Die Scores darauf heißen entsprechend
**Validierungsscores**.

Scikit-Learn macht diese Unterscheidung nicht mit, der Schlüssel heißt
weiterhin `'test_score'`. Im Code speichern wir die Werte deshalb in der
Variablen `validierungs_scores`.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

for max_tiefe in [1, 2, 3, 4, 5, 6]:
    modell = DecisionTreeClassifier(max_depth=max_tiefe, random_state=0)
    cv_results = # TODO: ???  (Kreuzvalidierung nur auf den Trainingsdaten)
    validierungs_scores = cv_results['test_score']
    mittelwert = np.mean(validierungs_scores)
    streuung = np.std(validierungs_scores)
    print(f'Baumtiefe {max_tiefe}, Validierungsscores: Mittelwert {mittelwert:.2f}, Standardabweichung {streuung:.2f}')

Diesmal liegt die Baumtiefe 3 mit einem mittleren Validierungsscore von 0.77
vorn, knapp vor den Tiefen 2 und 5 mit je 0.76. Wie im vorigen Abschnitt sind
die Unterschiede kleiner als die Schwankungen. Wir wählen die Tiefe 3,
trainieren diesen Baum auf allen Trainingsdaten und bewerten ihn ein einziges
Mal mit den Testdaten.

In [ ]:
finales_modell = DecisionTreeClassifier(max_depth=3, random_state=0)
finales_modell.fit(X_train, y_train)
accuracy_test = finales_modell.score(X_test, y_test)
print(f'Accuracy Testdaten finales Modell: {accuracy_test:.2f}')

Der Testscore von 0.72 liegt etwas unter dem besten Validierungsscore von 0.77.
Das passt zu unserer Überlegung vom Anfang dieses Abschnitts.

## Gittersuche mit GridSearchCV

Im vorigen Abschnitt haben wir die Baumtiefen mit einer for-Schleife verglichen
und die beste Tiefe selbst aus der Ausgabe abgelesen. Scikit-Learn nimmt uns
diese Arbeit ab. Die **Gittersuche** mit Kreuzvalidierung heißt dort
**GridSearchCV** und wird aus dem Modul `sklearn.model_selection` importiert.

Zunächst legen wir fest, welche Hyperparameter optimiert werden sollen und
welche Werte dafür in Betracht kommen. Technisch benötigen wir dafür ein
Dictionary, in dem die Schlüssel die Namen der Hyperparameter und die Werte
Listen der möglichen Einstellungen sind. In unserem Fall soll die Baumtiefe
`'max_depth'` des Entscheidungsbaums justiert werden. Wie zuvor in der
for-Schleife untersuchen wir die Baumtiefen 1 bis 6, die im folgenden Dictionary
`parameter_gitter` definiert werden.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Festlegung des Suchraumes
parameter_gitter = {'max_depth': [1, 2, 3, 4, 5, 6]}

Nun instanziieren wir ein neues `GridSearchCV`-Modell. Als erstes Argument
übergeben wir das eigentliche Modell, hier also den Entscheidungsbaum. Mit
`param_grid=` folgt das Dictionary mit den Hyperparametern und mit `cv=` wie
bei `cross_validate()` die Aufteilung für die Kreuzvalidierung. Weitere Details
stehen in der [Dokumentation
Scikit-Learn →
GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html#sklearn.model_selection.GridSearchCV).

In [ ]:
optimiertes_modell = # TODO: ???  (Gittersuche für einen Entscheidungsbaum mit dem Wert 0 für die Reproduzierbarkeit, dem Suchraum und der Fold-Aufteilung)

Mit der Methode `.fit()` wird die Gittersuche samt Kreuzvalidierung
durchgeführt. GridSearchCV übernimmt dabei genau das, was wir eben von Hand
programmiert haben. Jede Hyperparameter-Kombination wird auf den
Validierungsdaten bewertet, und das beste Modell wird abschließend erneut auf
den gesamten Trainingsdaten trainiert. Die Testdaten bleiben dabei unberührt.

In [ ]:
optimiertes_modell.fit(X_train, y_train)

Die Methoden `.score()` und `.predict()` funktionieren wie bei jedem anderen
Modell. Mit `.score()` bewerten wir das optimierte Modell auf den Testdaten.

In [ ]:
accuracy_test_optimiert = optimiertes_modell.score(X_test, y_test)
print(f'optimierter Entscheidungsbaum Accuracy Testdaten: {accuracy_test_optimiert:.2f}')

Zusätzlich zu den Standardmethoden wie `.fit()`, `.predict()` und `.score()`
können wir mit dem Attribut `best_params_` herausfinden, welche
Hyperparameter-Kombination am besten abgeschnitten hat. Das Attribut
`best_score_` liefert den zugehörigen mittleren Validierungsscore.

In [ ]:
# TODO: ???  (beste Hyperparameter-Kombination ausgeben)
print(f'Bester Validierungsscore: {optimiertes_modell.best_score_:.2f}')

Die Gittersuche wählt die Baumtiefe 3 mit einem mittleren Validierungsscore von
0.77. Das ist genau das Ergebnis, das wir im vorigen Abschnitt von Hand
ermittelt haben, und auch der Testscore von 0.72 stimmt überein.

Warum sprechen wir von einer Gittersuche? Normalerweise wollen wir nicht nur
einen Hyperparameter optimieren, sondern mehrere gleichzeitig. Beispielsweise
könnten wir neben der Baumtiefe auch die minimale Anzahl an Datenpunkten pro
Blatt (`min_samples_leaf`) optimieren. Dazu kombinieren wir jeden Wert von
`max_depth` mit jedem Wert von `min_samples_leaf` und testen alle
Kombinationen. So entsteht ein zweidimensionales Gitter, das die Gittersuche
systematisch durchläuft. Wir müssen lediglich das Dictionary entsprechend
erweitern. In diesem Beispiel werden 6 Baumtiefen und 3 Werte für
`min_samples_leaf` kombiniert, was zu insgesamt 6 × 3 = 18
Hyperparameter-Kombinationen führt. Da wir 5-fache Kreuzvalidierung verwenden,
werden dabei insgesamt 90 Modelle trainiert und bewertet.

In [ ]:
parameter_gitter = {
    'max_depth': [1, 2, 3, 4, 5, 6],
    'min_samples_leaf': [1, 2, 3]
}

optimiertes_modell = GridSearchCV(DecisionTreeClassifier(random_state=0), param_grid=parameter_gitter, cv=kfold)
optimiertes_modell.fit(X_train, y_train)
accuracy_test_optimiert = optimiertes_modell.score(X_test, y_test)

print(optimiertes_modell.best_params_)
print(f'Bester Validierungsscore: {optimiertes_modell.best_score_:.2f}')
print(f'optimierter Entscheidungsbaum Accuracy Testdaten: {accuracy_test_optimiert:.2f}')

Mit dem erweiterten Gitter wählt die Gittersuche die Baumtiefe 4 mit mindestens
3 Datenpunkten pro Blatt. Der beste Validierungsscore steigt dabei von 0.77 auf
0.81. Diese Wahl trifft die Gittersuche allein anhand der Validierungsscores.
Die Testdaten nutzen wir erst danach, und zwar einmal für das gewählte Modell.
Der Testscore von 0.76 liegt wieder unter dem besten Validierungsscore. Wir
dürfen ihn aber nicht mit dem Testscore 0.72 der ersten Gittersuche vergleichen,
um uns für eines der beiden Gitter zu entscheiden. Dann würden wir die
Testdaten wieder zur Auswahl nutzen. In der Praxis legen wir das Gitter deshalb
vorher fest und bewerten nur das endgültige Modell mit den Testdaten.

Bei größeren Datensätzen und komplexeren Modellen ist die Gittersuche ein
wertvolles Werkzeug. Da sie alle Kombinationen durchprobiert, wird sie bei einem
großen Gitter aber schnell rechenintensiv. Mit dem Argument `n_jobs=-1` verteilt
GridSearchCV die Berechnungen auf alle Prozessorkerne und verkürzt so die
Rechenzeit.

Eine Alternative zu GridSearchCV ist **RandomizedSearchCV**. Dieses Verfahren
testet nur eine zufällige Auswahl der Kombinationen. Das spart Zeit und liefert
oft trotzdem gute Ergebnisse. Mehr Details dazu stehen in der
[Dokumentation Scikit-Learn →
RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html#sklearn.model_selection.RandomizedSearchCV).

## Zusammenfassung und Ausblick

In diesem Kapitel haben wir erstmals systematisch Hyperparameter optimiert.
Dazu haben wir die Daten in drei Teile zerlegt: Trainingsdaten zum Trainieren,
Validierungsdaten zum Vergleichen der Hyperparameter und Testdaten für die
abschließende, unbefangene Bewertung. Die Gittersuche mit Kreuzvalidierung
nimmt uns das Trainieren und Vergleichen ab, sobald wir ihr ein Parametergitter
übergeben.

Eine Frage haben wir dabei die ganze Zeit übergangen. Wir haben immer die
Accuracy optimiert, also den Anteil der richtig zugeordneten Datenpunkte. Ob sie
für jede Aufgabe das richtige Gütemaß ist, haben wir nie gefragt. Für eine
Qualitätskontrolle kann sie eine schlechte Wahl sein.
Wenn nur 2 % aller Bauteile Ausschuss sind, erreicht ein Modell, das immer "in
Ordnung" vorhersagt, bereits 98 %. Im nächsten Kapitel lernen wir daher
verschiedene Gütemaße für Klassifikation und Regression kennen und klären, wann
welches Maß sinnvoll ist.

## Mini-Übungen

### Mini-Übung 1

Übertragen Sie das Vorgehen auf einen Datensatz aus der Fertigung. Die Datei
`3ddruck_gittersuche.csv` enthält 300 3D-Druckaufträge. Wir wollen mit einem
Entscheidungsbaum vorhersagen, ob ein Druck erfolgreich war (`Erfolgreich`:
ja/nein).

1. Lesen Sie die Datei ein. Verwenden Sie als Eingabedaten die Merkmale
   `Drucktemperatur (C)`, `Betttemperatur (C)` und
   `Druckgeschwindigkeit (mm/s)` und als Zielgröße das Merkmal `Erfolgreich`.
   Wählen Sie für Daten und Zielgröße andere Namen als im Fließtext, damit Sie
   die Beispieldaten nicht überschreiben.
2. Vergleichen Sie Entscheidungsbäume mit den Baumtiefen 1 bis 6 per
   Kreuzvalidierung mit fünf Folds. Mischen Sie die Daten vorher und sorgen Sie
   wie im Fließtext dafür, dass beide Klassen in jedem Fold im selben
   Verhältnis vertreten sind wie im gesamten Datensatz. Verwenden Sie für die
   Aufteilung und den Entscheidungsbaum `random_state=0`, damit Ihre
   Ergebnisse mit der Lösung vergleichbar sind. Geben Sie für jede Baumtiefe
   Mittelwert und Standardabweichung der Testscores aus. Lassen Sie sich
   zusätzlich die Trainingsscores berechnen und geben Sie deren Mittelwert aus.
3. Welche Baumtiefe schneidet am besten ab? Vergleichen Sie für die Tiefen 1
   und 6 die Trainingsscores mit den Testscores. Was deutet auf Underfitting
   hin, was auf Overfitting?

In [ ]:
# Code-Zelle

### Mini-Übung 2

Setzen Sie die vorherige Mini-Übung fort.

1. Teilen Sie die Druckaufträge wie im Fließtext in Trainings- und Testdaten
   auf. Verwenden Sie auch hier `random_state=0`.
2. Vergleichen Sie die Baumtiefen 1 bis 6 erneut per Kreuzvalidierung, diesmal
   nur auf den Trainingsdaten. Welche Baumtiefe wählen Sie?
3. Trainieren Sie einen Entscheidungsbaum mit dieser Baumtiefe auf allen
   Trainingsdaten und bewerten Sie ihn mit den Testdaten. Vergleichen Sie den
   Testscore mit dem besten Validierungsscore.

In [ ]:
# Code-Zelle

### Mini-Übung 3

Setzen Sie die vorherige Mini-Übung fort.

1. Wählen Sie die Baumtiefe diesmal automatisch mit einer Gittersuche aus.
   Erhalten Sie dieselbe Baumtiefe und denselben besten Validierungsscore wie
   in der vorherigen Mini-Übung?
2. Erweitern Sie die Suche um die minimale Anzahl an Druckaufträgen pro Blatt
   mit den Werten 1, 5 und 10. Wie viele Kombinationen werden getestet, und wie
   viele Modelle werden dabei in der Kreuzvalidierung trainiert und bewertet?
3. Geben Sie die beste Kombination und den besten Validierungsscore aus.
   Vergleichen Sie die Verbesserung des Validierungsscores gegenüber der ersten
   Gittersuche mit den Standardabweichungen aus der vorherigen Mini-Übung. Wie
   beurteilen Sie die Verbesserung? Bewerten Sie das gewählte Modell zum
   Schluss einmal mit den Testdaten.

In [ ]:
# Code-Zelle